# CA1 - Eoghan Murphy (123330861)

*Sri Lankan car prices*

## Business Problem

You live in Sri Lanka and you need to sell your car. But how much should you ask for it?

### Performance Measures

This is a regression problem on a dataset of nearly 20,000 adverts. As the dataset is large holdout may be more suitable than KFoldCV. I will build a dummy regression based on the mean of all prices.

## Imports

In [162]:
import pandas as pd
import numpy as np

from re import sub

from sklearn.model_selection import train_test_split

In [163]:
rng = np.random.RandomState(2)

## The Dataset

Each row contains 19 pieces of data:

1. `Title` -> Post title
2. `Sub_title` -> The time, date, and location of the posting
3. `Price` -> The price of the sale
4. `Brand` -> The brand of the car (Toyota, Micro, etc.)
5. `Model` -> The Model of the car (Panda, GT86, etc.)
6. `Edition` -> The version of the car's model
7. `Year` -> The year the car was made
8. `Condition` -> The condition the car is in
9. `Transmission` -> Whether automatic or manual
10. `Body` -> The type of the body of the car
11. `Fuel` -> The type of fuel it takes
12. `Capacity` -> The capacity of the engine
13. `Mileage` -> The distance the car has driven
14. `Location` -> The location of the sale
15. `Description` -> The description of the car
16. `Post_URL` -> The ad's URL
17. `Seller_name` -> The name of the seller
18. `Seller_type` -> The type of the membership the seller has on the site
19. `Published_date` -> The date and time the ad was posted

In [132]:
from os import path

base_dir = '.'
dataset_dir = path.join(base_dir, 'datasets')

In [133]:
df: pd.DataFrame = pd.read_csv(path.join(dataset_dir, 'dataset_vehicles.csv'))

In [134]:
# Capitalising the published_data column for convenience
df.rename(columns={'published_date': 'Published_date'}, inplace=True)

## Take a Cheeky Look

In [135]:
df.shape

(18938, 19)

In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18938 entries, 0 to 18937
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Title           18938 non-null  object
 1   Sub_title       18938 non-null  object
 2   Price           18938 non-null  object
 3   Brand           18938 non-null  object
 4   Model           18938 non-null  object
 5   Edition         13908 non-null  object
 6   Year            18938 non-null  int64 
 7   Condition       18938 non-null  object
 8   Transmission    18938 non-null  object
 9   Body            17043 non-null  object
 10  Fuel            18938 non-null  object
 11  Capacity        18938 non-null  object
 12  Mileage         18938 non-null  object
 13  Location        18938 non-null  object
 14  Description     18938 non-null  object
 15  Post_URL        18938 non-null  object
 16  Seller_name     18938 non-null  object
 17  Seller_type     18938 non-null  object
 18  Publis

Observations:

- `Price` & `Capacity` & `Mileage` are objects. Why?
- `Condition` maybe ordinal?
- `Body` is missing values. Replace with what?
- `Edition` is missing values. Replace with what?
- `Fuel` is binary maybe?

In [137]:
df.head()

,Title,Sub_title,Price,Brand,Model,Edition,Year,Condition,Transmission,Body,Fuel,Capacity,Mileage,Location,Description,Post_URL,Seller_name,Seller_type,Published_date
0,Micro Panda Auto 2019 for sale,"Posted on 02 Jul 6:27 am, Kesbewa, Colombo","Rs 2,795,000",Micro,Panda,Auto,2019,Used,Automatic,Hatchback,Petrol,"1,000 cc","14,000 km","Kesbewa, Colombo",â¤ Auto Auto Autoâ¤ Micro Panda CBH-xxxx 201...,https://ikman.lk/en/ad/micro-panda-auto-2019-f...,Sarath,Premium-Member,2021-07-02 06:27:00
1,Toyota GT86 Company Maintained 2013 for sale,"Posted on 15 Jun 7:38 pm, Rajagiriya, Colombo","Rs 11,500,000",Toyota,GT86,Company Maintained,2013,Used,Automatic,CoupÃ©/Sports,Petrol,"2,000 cc","27,000 km","Rajagiriya, Colombo",Toyota Gt86 Company maintained2013 Manufacture...,https://ikman.lk/en/ad/toyota-gt86-company-mai...,D&D Auto Parts,Premium-Member,2021-06-15 19:38:00
2,"Toyota IST FL , GRADE 2003 for sale","Posted on 05 Jul 1:36 pm, Marawila, Puttalam","Rs 3,300,000",Toyota,IST,"FL , GRADE",2003,Used,Automatic,Hatchback,Petrol,"1,300 cc","150,000 km","Marawila, Puttalam",TOYOTA ISTFL GRADEKD ; >>>> NUMBERYOM 2003RAGE...,https://ikman.lk/en/ad/toyota-ist-fl-grade-200...,Bihan Enterprises,Premium-Member,2021-07-05 13:36:00
3,Toyota Allion 240 2002 for sale,"Posted on 05 Jul 1:33 pm, Peradeniya, Kandy","Rs 4,450,000",Toyota,Allion,240,2002,Used,Automatic,Saloon,Petrol,"1,500 cc","124,000 km","Peradeniya, Kandy",Toyota Allion 240CP KA-4xxxManufactured 2002Re...,https://ikman.lk/en/ad/toyota-allion-240-2002-...,Free Bird Media (Pvt) Ltd,Premium-Member,2021-07-05 13:33:00
4,Micro Panda LC 1.0 2015 for sale,"Posted on 05 Jul 1:30 pm, Veyangoda, Gampaha","Rs 2,173,000",Micro,Panda,LC 1.0,2015,Used,Manual,Hatchback,Petrol,"1,000 cc","15,800 km","Veyangoda, Gampaha","CAJ XXXX ,Silver colour ,1st Owner , Full Opti...",https://ikman.lk/en/ad/micro-panda-lc-10-2015-...,Harsha Anuradhi,Premium-Member,2021-07-05 13:30:00


Observation:

- `Price` -> Converted to int, remove 'Rs'
- `Body` -> Body seems to have character encoding errors
- `Capacity` -> Converted to int, remove 'cc'
- `Mileage` -> Converted to int, remove 'km'
- `Description` -> Character encoding issues
- `Seller_name` -> Some are people, some are companies, could be indicative
- `Seller_type` -> Ordinal?

In [138]:
df.describe(include='all')

,Title,Sub_title,Price,Brand,Model,Edition,Year,Condition,Transmission,Body,Fuel,Capacity,Mileage,Location,Description,Post_URL,Seller_name,Seller_type,Published_date
count,18938,18938,18938,18938,18938,13908,18938.000000,18938,18938,17043,18938,18938,18938,18938,18938,18938,18938,18938,18938
unique,10329,17014,1831,57,516,6336,NaN,3,4,7,6,432,3850,198,17075,17911,6103,1,15060
top,Suzuki Alto 2015 for sale,"Posted on 03 Mar 11:21 am, Colombo 3, Colombo","Rs 650,000",Toyota,Alto,G Grade,NaN,Used,Automatic,Hatchback,Petrol,"1,500 cc","100,000 km","Kohuwala, Colombo",* Leasing can be arranged with your requiremen...,https://ikman.lk/en/ad/toyota-pixis-g-intelige...,LB Finance PLC,Premium-Member,2021-03-03 11:21:00
freq,184,63,92,5762,962,172,NaN,17991,11412,6712,13823,3861,450,1333,151,4,512,18938,69
mean,NaN,NaN,NaN,NaN,NaN,NaN,2007.718344,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,11.640139,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1927.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,2003.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,2012.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,2016.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Observations:

- High frequency of `Title` mode. Maybe duplicate or just reposting
- High frequency of `Sub_title` mode.
- Min `Year` is 1927. Error?.
  - Looked on the website and found a seemingly plausible [ad](https://ikman.lk/en/ad/austin-7-tourer-1936-for-sale-kegalle) for a car from 1936 so I deemed it valid data
- High frequency of `Description` mode. Likely duplicate
- `Post_url` has duplicate entries, is the whole row duplicated?
- `Seller_type` has only 1 value, so has no effect on target, so drop

In [139]:
df.duplicated().sum()

np.int64(289)

Observation:

- Quite a number of duplicates in a large number dataset, just remove.
- Could be some more duplicates but with different URLs and posting times for sales that were relisted

In [140]:
df[df['Title'] == 'Suzuki Alto 2015 for sale'].duplicated().sum()

np.int64(3)

Very few duplicates so different people selling the same car.

In [141]:
len(
    df[
        (df['Sub_title'] == 'Posted on 03 Mar 11:21 am, Colombo 3, Colombo')
        & (df['Seller_name'] == 'LB Finance PLC')
    ]
)

63

A number of ads produced by the same company at the same time for different cars, so not a problem.

In [142]:
df.loc[df['Description'] == df['Description'].mode().iloc[0], 'Seller_name'].unique()

array(['LB Finance PLC', 'LB Finance Plc'], dtype=object)

All from the same company so clearly just a template description.

<sub>Noticed a potential error, `PLC` should == `Plc`</sub>

In [143]:
df.loc[df.index[np.where(df.groupby('Post_URL').size() > 1)]].head(1)

,Title,Sub_title,Price,Brand,Model,Edition,Year,Condition,Transmission,Body,Fuel,Capacity,Mileage,Location,Description,Post_URL,Seller_name,Seller_type,Published_date
6,Micro Panda Auto 2017 for sale,"Posted on 02 Jul 6:27 am, Kesbewa, Colombo","Rs 2,795,000",Micro,Panda,Auto,2017,Used,Automatic,Hatchback,Petrol,"1,000 cc","22,000 km","Kesbewa, Colombo",â¤ Auto Auto Autoâ¤ Brandnew Condition Assur...,https://ikman.lk/en/ad/micro-panda-auto-2017-f...,Sarath,Premium-Member,2021-07-02 06:27:00


In [144]:
df.loc[
    df['Post_URL'] == 'https://ikman.lk/en/ad/micro-panda-auto-2017-for-sale-colombo-5'
]

,Title,Sub_title,Price,Brand,Model,Edition,Year,Condition,Transmission,Body,Fuel,Capacity,Mileage,Location,Description,Post_URL,Seller_name,Seller_type,Published_date
6,Micro Panda Auto 2017 for sale,"Posted on 02 Jul 6:27 am, Kesbewa, Colombo","Rs 2,795,000",Micro,Panda,Auto,2017,Used,Automatic,Hatchback,Petrol,"1,000 cc","22,000 km","Kesbewa, Colombo",â¤ Auto Auto Autoâ¤ Brandnew Condition Assur...,https://ikman.lk/en/ad/micro-panda-auto-2017-f...,Sarath,Premium-Member,2021-07-02 06:27:00
5069,Micro Panda Auto 2017 for sale,"Posted on 23 Jul 7:54 am, Kesbewa, Colombo","Rs 2,795,000",Micro,Panda,Auto,2017,Used,Automatic,Hatchback,Petrol,"1,000 cc","22,000 km","Kesbewa, Colombo",â¤ Auto Auto Autoâ¤ Brandnew Condition Assur...,https://ikman.lk/en/ad/micro-panda-auto-2017-f...,Sarath,Premium-Member,2021-07-23 07:54:00
10496,Micro Panda Auto 2017 for sale,"Posted on 01 Aug 7:38 pm, Kesbewa, Colombo","Rs 2,995,000",Micro,Panda,Auto,2017,Used,Automatic,Hatchback,Petrol,"1,000 cc","22,000 km","Kesbewa, Colombo",â¤ Auto Auto Autoâ¤ Brandnew Condition Assur...,https://ikman.lk/en/ad/micro-panda-auto-2017-f...,Sarath,Premium-Member,2021-08-01 19:38:00


Relisting the same ad so same URL

In [145]:
df['Condition'].unique(), df['Fuel'].unique()

(array(['Used', 'New', 'Reconditioned'], dtype=object),
 array(['Petrol', 'Hybrid', 'Diesel', 'CNG', 'Electric', 'Other fuel type'],
       dtype=object))

- Could use ordinal encoding as values are ordered.
- Fuel has more than 2 so not binary

### Final Notes

- Large dataset so likely holdout over KFoldCV
- Need to sort out nulls in `Body`
- Need to convert `Price`, `Capacity` & `Mileage` to `int`
- Convert `Published_date` to number of seconds since Jan 1<sup>st</sup> 1970.
- `Condition` could be encoded using `OrdinalEncoding`
- Character encoding issues in `Body`, `Description` and possibly other `string` fields
- `Seller_name` has companies and people
  - `Company` being a bool if true if company could be a good feature to add
- `Location` has city and country, could split into 2 columns.
- Need to make sure all strings match regardless of case so maybe lowercase everything?
- `Seller_type` has one value so drop as makes no difference

## Cleanup

**Don't impute here**

### Fix Types

In [146]:
def make_int(col: str, text: str):
    df[col] = df[col].apply(lambda row: sub(rf'{text}|,', '', row)).astype('int64')

In [147]:
make_int('Price', 'Rs ')
make_int('Capacity', ' cc')
make_int('Mileage', ' km')

In [148]:
from datetime import datetime

df['Published_date'] = df['Published_date'].apply(
    lambda row: round(
        (
            datetime.strptime(row, r'%Y-%m-%d %H:%M:%S') - datetime(1970, 1, 1)
        ).total_seconds()
    )
)

In [150]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18938 entries, 0 to 18937
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Title           18938 non-null  object
 1   Sub_title       18938 non-null  object
 2   Price           18938 non-null  int64 
 3   Brand           18938 non-null  object
 4   Model           18938 non-null  object
 5   Edition         13908 non-null  object
 6   Year            18938 non-null  int64 
 7   Condition       18938 non-null  object
 8   Transmission    18938 non-null  object
 9   Body            17043 non-null  object
 10  Fuel            18938 non-null  object
 11  Capacity        18938 non-null  int64 
 12  Mileage         18938 non-null  int64 
 13  Location        18938 non-null  object
 14  Description     18938 non-null  object
 15  Post_URL        18938 non-null  object
 16  Seller_name     18938 non-null  object
 17  Seller_type     18938 non-null  object
 18  Publis

In [151]:
df[
    (df['Body'].str.contains(r'[^a-zA-z\s0-9/]', regex=True)) & (df['Body'].notna())
].head()

,Title,Sub_title,Price,Brand,Model,Edition,Year,Condition,Transmission,Body,Fuel,Capacity,Mileage,Location,Description,Post_URL,Seller_name,Seller_type,Published_date
1,Toyota GT86 Company Maintained 2013 for sale,"Posted on 15 Jun 7:38 pm, Rajagiriya, Colombo",11500000,Toyota,GT86,Company Maintained,2013,Used,Automatic,CoupÃ©/Sports,Petrol,2000,27000,"Rajagiriya, Colombo",Toyota Gt86 Company maintained2013 Manufacture...,https://ikman.lk/en/ad/toyota-gt86-company-mai...,D&D Auto Parts,Premium-Member,1623785880
94,Mercedes Benz C200 AMG Coupe 2018 for sale,"Posted on 04 Jul 11:32 am, Dehiwala, Colombo",24000000,Mercedes Benz,C200,AMG Coupe,2018,Used,Automatic,CoupÃ©/Sports,Petrol,1500,28000,"Dehiwala, Colombo",ððð«ððððð¬ ððð§ð...,https://ikman.lk/en/ad/mercedes-benz-c200-amg-...,Monarch-Automobile,Premium-Member,1625398320
195,Mercedes Benz CLA 200 AMG PREMIUM PLUS 2019 fo...,"Posted on 14 Jun 1:03 pm, Negombo, Gampaha",17750000,Mercedes Benz,CLA 200,AMG PREMIUM PLUS,2019,Used,Tiptronic,CoupÃ©/Sports,Petrol,1400,8500,"Negombo, Gampaha",BENZ CLA 200 AMG Premium Plus 20198000miles do...,https://ikman.lk/en/ad/mercedes-benz-cla-200-a...,Dineeth Auto Traders,Premium-Member,1623675780
297,Mercedes Benz CLA 200 AMG PREMIUM PLUS 2019 fo...,"Posted on 05 Jul 6:19 am, Kohuwala, Colombo",17990000,Mercedes Benz,CLA 200,AMG PREMIUM PLUS,2019,Used,Automatic,CoupÃ©/Sports,Petrol,1200,15000,"Kohuwala, Colombo",2019 MERCEDES-BENZ CLA 200 AMG PREMIUM PLUS Ye...,https://ikman.lk/en/ad/mercedes-benz-cla-200-a...,Suren Auto Mart,Premium-Member,1625465940
501,Daihatsu Charade G11 1987 for sale,"Posted on 04 Jul 2:34 pm, Kesbewa, Colombo",575000,Daihatsu,Charade,G11,1987,Used,Manual,CoupÃ©/Sports,Petrol,1000,118245,"Kesbewa, Colombo",Hodama dawana tatwaye ata. Atula pita company ...,https://ikman.lk/en/ad/daihatsu-charade-g11-19...,sanka,Premium-Member,1625409240


Only character problem was `CoupÃ©/Sports`, so I got some of the values for the cars brand and model, and looked them up to find the body type listed as `Coupe` so I'm replacing them with `Coupe/Sports`.

Further looked up the cars on Ikman and found `Coupé/Sports` as an filter option for body, but I'm going to replace it with `Coupe/Sports` so I don't have to worry about `é`.

In [152]:
df.loc[
    (df['Body'].str.contains(r'[^a-zA-z\s0-9/]', regex=True)) & (df['Body'].notna()),
    'Body',
] = 'Coupe/Sports'

In [153]:
df.loc[
    (df['Description'].str.contains(r'[^a-zA-z0-9\s]', regex=True)), 'Description'
].head(1)[0]

'â\x9d¤ Auto Auto Autoâ\x9d¤ Micro Panda CBH-xxxx 2019â\x9d¤ Brand New Condition Assured â\x9d¤ Genuine low milledge 14000 KMsâ\x9d¤ à·\x83à¶\xadà·\x8aâ\x80\x8dà¶º à¶\x85à¶©à·\x94 à¶°à·\x8fà·\x80à¶±à¶º à¶\x9aà·\x93à¶¸à·\x93 14000â\x9d¤ Dual Air bag ABS Auto  - Highest grade of Panda Familyâ\x9d¤ à¶©à·\x94à·\x80à¶½à·\x8a à¶\x91à¶ºà·\x8f à¶¶à·\x91à¶\x9cà·\x8a à¶\x92à¶¶à·\x93à¶\x91à·\x83à·\x8a à¶\x94à¶§à·\x9d - à¶´à·\x90à¶±à·\x8aà¶©à·\x8f à¶´à·\x80à·\x94à¶½à·\x9a à¶\x89à·\x84à¶½à¶¸ à¶¸à·\x8fà¶¯à·\x92à¶½à·\x92à¶ºâ\x9d¤ 100% Company maintained â\x9d¤ à¶±à¶©à¶\xadà·\x8aà¶\xadà·\x94à·\x80 à·\x83à¶¸à·\x8fà¶\x9cà¶¸à·\x92à¶±à·\x8a à¶´à¶¸à¶±à·\x92 â\x9d¤ All service and emission recordsâ\x9d¤ à·\x83à·\x92à¶ºà¶½à·\x94 à¶±à¶©à¶\xadà·\x8aà¶\xadà·\x94 à¶¯à·\x94à¶¸à·\x8a à·\x80à·\x8fà¶»à·\x8aà¶®à·\x8f â\x9d¤ 100% Accident free - Guranteed â\x9d¤ 100% à¶\x85à¶±à¶\xadà·\x94à¶»à·\x94 à¶»à·\x84à·\x92à¶\xadà¶ºà·\x92 - à·\x83à·\x84à¶\xadà·\x92à¶\x9aà¶ºà·\x92 â\x9d¤ First Paint - No touchupsâ\x9d¤ à¶´à·\x8aâ

Decided to drop `Description` for now, because I can't figure out how to decode it yet. Might come back and just take words out of it like if it mentions air bags, etc.

In [154]:
# Checking for character encoding errors in other features
for feature in [
    # 'Title',
    # 'Sub_title',
    # 'Brand',
    # 'Model',
    'Edition',
    # 'Condition',
    # 'Transmission',
    # 'Fuel',
    # 'Location',
    'Seller_name',
    # 'Seller_type',
    # 'Published_date'
]:
    print(
        df.loc[
            (df[feature].str.contains(r"[^a-zA-z0-9\s,\.\-'/|+()%\":]", regex=True))
            & (df[feature].notna()),
            feature,
        ]
    )
    print()

1402                                   Hirukiâs car
1403                                   Hirukiâs car
1648                               C180 CoupÃ© Sport 
2186                                   XLT Turbo 4Ã4
2286         à¶¸à·à¶¯à¶½à· à·à¶¯à·à·à·à¶ºà¶à¶§
3972                                11à·à·âà¶»à·
4793                                             4Ã4
6301                                             Ä¢ +
8437                                    bolero LX 4*4
8714                                   Hirukiâs car
10810                       ð-ðð©ð¨ð«ð­
11222                                            ð
11563                        à·à¶¯à·à·à·à¶ºà¶à·
12275                                          R~Line
12661                                         KF-####
15620                                            ð
16554                                        CHR-****
17024    j style à·à·à¶´à·à¶»à·à¶­à¶­à·à·à¶ºà·
17697                       

Decided to drop `Title`, as there is errors but it is made up of other columns therefore no new information, so not worth fixing.
Decided to also drop `Sub_title` as no new information.

`Brand`, `Model`, `Condition`, `Transmission`, `Fuel`, `Location`, `Seller_type` and `Published_date` had no errors in them.

Only need to deal with `Edition` and `Seller_name`.

In [155]:
def get_errors(feature):
    return df.loc[
        (df[feature].str.contains(r"[^a-zA-z0-9\s,\.\-'/|+()%\":&!]", regex=True))
        & (df[feature].notna())
    ]

In [156]:
get_errors('Edition')

,Title,Sub_title,Price,Brand,Model,Edition,Year,Condition,Transmission,Body,Fuel,Capacity,Mileage,Location,Description,Post_URL,Seller_name,Seller_type,Published_date
1402,Suzuki Wagon R FZ Hirukiâs car 2017 for sale,"Posted on 04 Jul 5:08 pm, Kohuwala, Colombo",3890000,Suzuki,Wagon R FZ,Hirukiâs car,2017,Used,Automatic,Hatchback,Hybrid,650,67000,"Kohuwala, Colombo",1st owner all service record available origina...,https://ikman.lk/en/ad/suzuki-wagon-r-fz-hiruk...,Hiruki Auto Mart,Premium-Member,1625418480
1403,Suzuki Wagon R Stingray Hirukiâs car 2018 fo...,"Posted on 04 Jul 5:07 pm, Kohuwala, Colombo",4390000,Suzuki,Wagon R Stingray,Hirukiâs car,2018,Used,Automatic,Hatchback,Hybrid,650,38900,"Kohuwala, Colombo",1st owner original painting service record ava...,https://ikman.lk/en/ad/suzuki-wagon-r-stingray...,Hiruki Auto Mart,Premium-Member,1625418420
1648,Mercedes Benz CLA 180 C180 CoupÃ© Sport 2018 f...,"Posted on 30 Jun 8:50 pm, Nugegoda, Colombo",15900000,Mercedes Benz,CLA 180,C180 CoupÃ© Sport,2018,Used,Automatic,Coupe/Sports,Petrol,1600,52000,"Nugegoda, Colombo",Mercedes Benz C180 CoupÃ© Sport Company Brand ...,https://ikman.lk/en/ad/mercedes-benz-cla-180-c...,Riyoga Motors,Premium-Member,1625086200
2186,Ford Ranger XLT Turbo 4Ã4 2019 for sale,"Posted on 27 Jun 5:30 pm, Kohuwala, Colombo",16700000,Ford,Ranger,XLT Turbo 4Ã4,2019,Used,Automatic,SUV / 4x4,Diesel,3200,31000,"Kohuwala, Colombo",Ford ranger XLT Turbo Y.O.M: 2019Engine: 3200c...,https://ikman.lk/en/ad/ford-ranger-xlt-turbo-4...,PRIME HOLDING,Premium-Member,1624815000
2286,Toyota Corolla à¶¸à·à¶¯à¶½à· à·à¶¯à·à·à·...,"Posted on 27 Jun 7:08 am, Kurunegala, Kurunegala",400000,Toyota,Corolla,à¶¸à·à¶¯à¶½à· à·à¶¯à·à·à·à¶ºà¶à¶§,1984,Used,Manual,Station wagon,Petrol,1300,246360,"Kurunegala, Kurunegala",à¶¸à·à¶¯à¶½à· à·à¶¯à·à·à·à¶ºà¶à¶§,https://ikman.lk/en/ad/toyota-corolla-mudl-hdi...,D.M HIRUNI NAWODYA,Premium-Member,1624777680
3972,Toyota Corolla 11à·à·âà¶»à· 1992 for sale,"Posted on 28 May 4:36 pm, Kurunegala, Kurunegala",370000,Toyota,Corolla,11à·à·âà¶»à·,1992,Used,Manual,Saloon,Diesel,1100,105874,"Kurunegala, Kurunegala",à¶´à·à¶»à¶±à· à·à·à¶±à¶ºà¶à· à·à·à¶ºà¶...,https://ikman.lk/en/ad/toyota-corolla-11shri-1...,upul lakmal,Premium-Member,1622219760
4793,Kia Carens 4Ã4 2003 for sale,"Posted on 09 May 7:43 am, Piliyandala, Colombo",2200000,Kia,Carens,4Ã4,2003,Used,Manual,Station wagon,Petrol,2200,88000,"Piliyandala, Colombo","Kia / Carens-2003,6 seater,A / C,Original all...",https://ikman.lk/en/ad/kia-carens-4x4-2003-for...,roshan indika,Premium-Member,1620546180
6301,Toyota Allion Ä¢ + 2003 for sale,"Posted on 19 Jul 7:45 am, Nittambuwa, Gampaha",5250000,Toyota,Allion,Ä¢ +,2003,Used,Automatic,Saloon,Petrol,1500,180000,"Nittambuwa, Gampaha","Accident free , no any damage, original body c...",https://ikman.lk/en/ad/toyota-allion-g-2003-fo...,Ranaweera Enterprises,Premium-Member,1626680700
8437,Mahindra Bolero LX 4*4 2003 for sale,"Posted on 08 Aug 11:57 am, Nittambuwa, Gampaha",2790000,Mahindra,Bolero,bolero LX 4*4,2003,Used,Manual,SUV / 4x4,Diesel,2500,120625,"Nittambuwa, Gampaha",MAHINDRA BOLERO LX 4*4 2003 manufactured 2003 ...,https://ikman.lk/en/ad/mahindra-bolero-lx-44-2...,Sampath Dambadeniya,Premium-Member,1628423820
8714,Toyota Yaris Hirukiâs car 2006 for sale,"Posted on 08 Aug 8:06 am, Kohuwala, Colombo",4350000,Toyota,Yaris,Hirukiâs car,2006,Used,Automatic,Saloon,Petrol,1300,112500,"Kohuwala, Colombo",1st owner duplicate book ( 8 years with Leesi...,https://ikman.lk/en/ad/toyota-yaris-hirukis-ca...,Hiruki Auto Mart,Premium-Member,1628409960


In [157]:
len(get_errors('Seller_name')['Seller_name'].unique())

73

Too many different values with errors so can't research each one, need to find another method to fix them.

In [158]:
get_errors('Seller_name').iloc[8]['Seller_name']

'Sell Fast | Adz King à¶¯à·\x90à¶±à·\x8aà·\x80à·\x93à¶¸à·\x8a à¶\x86à¶ºà¶\xadà¶±à¶º'

In [159]:
str_df = df.select_dtypes(include='object')
df[str_df.columns] = str_df.apply(lambda row: row.str.lower())

## Split dataset

In [166]:
numerical_features: list[str] = ['Year', 'Capacity', 'Mileage', 'Published_date']

nominal_features: list[str] = [
    'Brand',
    'Model',
    'Condition',
    'Transmission',
    'Body',
    'Fuel',
    'Location',
]

X = df[numerical_features + nominal_features]
y = df.Price

In [165]:
X_train: pd.DataFrame
X_test: pd.DataFrame
y_train: pd.Series
y_test: pd.Series

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rng)

## EDA

## Feature Engineering

## Feature Selection

## Preprocessing

## Model Selection

## Error Estimation